In [1]:
# 全局设置
import datetime as dt
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings('ignore', category=PerformanceWarning)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False# 正确显示负号
from IPython.display import HTML

from QuantStudio import __QS_MainPath__

# 截面因子测试

截面因子测试用于评估因子对证券截面收益的预测能力，是量化因子研究中最核心的环节之一。QuantStudio 的截面因子测试功能位于 `BackTest.SectionFactor` 子模块，建立在回测基本框架之上。

> **前置阅读**：回测框架的整体架构、`BTNode`/`BTReport` 基类、执行流程（三层嵌套 `Cache → Context → Engine`）以及"算子-节点分离"设计模式，请先参阅 **[基本框架](基本框架.ipynb)**。计算图与引擎的基础知识请参阅 **[计算图框架](../Core/计算图框架.ipynb)** 和 **[计算引擎](../Core/计算引擎.ipynb)**。

## 模块概览

`QuantStudio.BackTest.SectionFactor` 提供以下测试组件：

| 组件 | 算子（计算层） | 回测节点（报告层） | 功能 |
|------|---------------|-------------------|------|
| Rank IC | `CalcIC` | `IC` | 因子值与下期收益率的截面秩相关性 |
| 风险调整 IC | `CalcRiskAdjustedIC` | `IC` | 对风险因子正交化后的 IC |
| IC 衰减 | `CalcIC`（变周期） | `ICDecay` | IC 随回溯期数增加的变化 |
| 分位数组合 | `makeQuantilePortfolio` | `MultiPortfolio` | 按因子值分组构建投资组合 |
| 因子换手率 | `CalcFactorTurnover` | `FactorTurnover` | 前后两期因子值的截面相关性 |
| 截面相关性 | `CalcSectionCorrelation` | `SectionCorrelation` | 多因子两两之间的截面相关性 |
| Fama-MacBeth 回归 | `CalcFamaMacBethRegression` | `FamaMacBethRegression` | 剥离风险因子后的因子收益率 |

每个组件遵循"算子 + 回测节点"两层架构：算子（`PanelOperator` / `SectionOperator` 子类）封装纯计算逻辑，回测节点（`BTNode` 子类）负责统计汇总和 HTML 报告生成。

## 理论基础

### Rank IC

往期因子值和当期收益率的秩相关性。通常采用的相关系数为 Spearman 相关系数。记收益率的横截面向量 $R_{i,t}$ 的截面排名记为 $Q_{i,t}$，在 $t-1$ 时刻因子暴露的横截面向量 $\beta_{i,t}$ 的截面排名记为 $P_{i,t}$，那么 $t$ 时刻的 Rank IC 定义为：

$$
\rho\left(Q_t, P_{t-1}\right) = \frac{\sum\limits_{i=1}^{N} \left(Q_{i,t} - \bar{Q}_t\right)\left(P_{i,t-1} - \bar{P}_{t-1}\right)}{\sqrt{\sum\limits_{i=1}^{N} \left(Q_{i,t} - \bar{Q}_t\right)^2} \cdot \sqrt{\sum\limits_{i=1}^{N} \left(P_{i,t-1} - \bar{P}_{t-1}\right)^2}}
$$

其中：

$$
\begin{aligned}
& \bar{Q}_{t} = \frac{1}{N}\sum\limits_{i=1}^{N}Q_{i,t} \\
& \bar{P}_{t-1} = \frac{1}{N}\sum\limits_{i=1}^{N}P_{i,t-1}
\end{aligned}
$$

另外还可以计算**行业调整的 IC**：扣除行业收益率后的 Rank IC。对于股票 i，其收益率 $R_{i,t}$ 的行业调整值定义为 $\tilde{R}_{i,t}=R_{i,t}-R_{I,t}$，其中 $R_{I,t}$ 为行业 I 在时段 $[t-1,t]$ 的收益率。

### IC 的衰减

因子值和收益率关于日期间隔的 IC 衰减情况。对于因子 j，分别计算 $R_t$ 同 $\beta_{j,t-1},\beta_{j,t-2},\ldots,\beta_{j,t-K}$ 的 Rank IC，考察 Rank IC 序列关于时间间隔 $(1,2,\ldots,K)$ 的变化情况。

### 分位数组合

按因子值排序分组后形成的投资组合。对于目标因子，在时刻 t，首先剔除因子值缺失和不符合筛选条件的证券，对剩下的证券按照因子暴露进行排序，把排序后的证券平均分成 K 组（一般取 5 或者 10 组），每一组构建投资组合（通常为等权或者按照某个因子值加权），测试给定时间段每一组合的表现情况。

### 因子换手率

当期因子值和上期因子值横截面上的线性相关系数。对于给定因子，因子暴露的横截面向量 $\beta_{i,t}$ 的截面排名记为 $P_{i,t}$，日期 t 和上一日期 t-1 之间的换手率定义为：

$$
\rho\left(P_t, P_{t-1}\right) = \frac{\sum\limits_{i=1}^{N} \left(P_{i,t} - \bar{P}_t\right)\left(P_{i,t-1} - \bar{P}_{t-1}\right)}{\sqrt{\sum\limits_{i=1}^{N} \left(P_{i,t} - \bar{P}_t\right)^2} \cdot \sqrt{\sum\limits_{i=1}^{N} \left(P_{i,t-1} - \bar{P}_{t-1}\right)^2}}
$$

### Fama-MacBeth 回归

剥离了给定风险因子后的因子收益率。给定 K 个风险因子（一般取市值、Beta、行业等），记股票 i 在这些风险因子上的暴露为 $\beta_{i,1,t},\beta_{i,2,t},\ldots,\beta_{i,K,t}$，对于目标因子，在时刻 t，同风险因子一起与收益率向量 $\mathbf{R}_t$ 做横截面回归：

$$
R_{i,t} = f^*_{t}\beta^*_{i,t-1} + f_{1,t}\cdot \beta_{i,1,t-1}+ \cdots +f_{K,t}\cdot \beta_{i,K,t-1} + \varepsilon_{i,t}
$$

估计的回归系数 $f^*_{t}$ 即为剥离了这 K 个风险因子后目标因子在时段 t 的收益。

### 因子相关性

两个因子暴露的相关性。取时刻 t 的因子 i 和因子 j 的因子暴露 $\beta_{i,t}$、$\beta_{j,t}$，在横截面上计算相关性，方法可以选择：spearman、pearson、kendall 以及 factor-score correlation。

## Rank IC 分析

### CalcIC — IC 计算算子

`CalcIC` 是一个 `PanelOperator`（面板算子），对每个时点计算因子值与下期收益率的截面秩相关性。

**构造参数**：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `descriptor_ids` | `List[str]` | — | 依赖因子的截面 ID 序列 |
| `lookback` | `int` | `31` | 在时间标尺上的回溯期数 |
| `period_lookback` | `int` | `1` | 在计算标尺上的回溯期数（月度 IC 取 1 表示用上月因子值） |
| `corr_method` | `Literal["spearman","pearson","kendall"]` | `"spearman"` | 相关性计算方法 |

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `*x` | `Factor` | 待测试的因子列表 |
| `price` | `Factor` | 证券价格/净值因子，用于计算收益率 |
| `mask` | `Optional[Factor]` | 筛选条件因子，值为 1 的截面才参与计算 |
| `cat_data` | `Optional[Factor]` | 类别因子（如行业），非 None 时进行行业调整 |
| `weight` | `Optional[Factor]` | 权重因子，计算类别收益率时使用 |
| `factor_name_list` | `Optional[List[str]]` | 测试因子名称列表，默认自动生成 |
| `factor_args` | `dict` | 传递给 IC 因子的参数（如 `CalcDTRuler` 指定计算时点） |

**返回值**：产生一个复合类型因子，每行数据为 `(IC, Breadth)` 元组。

### IC — 回测报告节点

`IC` 是继承自 `BTNode` 的回测节点，接收 `CalcIC` 产生的因子作为依赖，汇总统计并生成 HTML 报告。

**参数（`__QS_ArgClass__`）**：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"IC"` | 节点名称（冻结） |
| `FactorNameList` | `Optional[List[str]]` | `None` | 因子名称列表，None 时自动从数据中获取 |
| `RollingAvgPeriod` | `int` | `12` | IC 移动平均的期数 |
| `GenReport` | `bool` | `False` | 是否自动生成报告 |

**输出数据结构**（`backward_compute` 返回的字典）：

| 键 | 类型 | 说明 |
|------|------|------|
| `"截面宽度"` | `DataFrame` | 每期参与计算的截面样本数 |
| `"IC"` | `DataFrame` | 每期每因子的 IC 值 |
| `"IC的移动平均"` | `DataFrame` | IC 的滚动平均值 |
| `"统计数据"` | `DataFrame` | 汇总统计：平均值、标准差、IC_IR、t 统计量、胜率等 |

### 示例代码

```python
from QuantStudio.BackTest.SectionFactor.IC import CalcIC, IC

# 创建 IC 算子并作用于测试因子
FactorIC = CalcIC(
    descriptor_ids=SectionIDs,
    lookback=31,
    period_lookback=1,
    corr_method="spearman"
)(
    *FactorList,
    price=Price,
    mask=Mask,
    cat_data=Industry,
    factor_args={"CalcDTRuler": BalanceDTs}
)

# 创建 IC 回测节点
ICNode = IC(FactorIC, args={
    "RollingAvgPeriod": 2,
    "GenReport": True
})
```

报告输出包含每个因子的 IC 时序柱状图、移动平均线、截面宽度曲线，以及汇总统计表（IC 均值、标准差、IC_IR、t 统计量等）。

## 截面相关性

### CalcSectionCorrelation — 截面相关性计算算子

`CalcSectionCorrelation` 是一个 `SectionOperator`，计算多个因子两两之间的截面相关性矩阵（每期一个值）。

**构造参数**：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `descriptor_ids` | `List[str]` | — | 截面 ID 序列 |
| `corr_method` | `Literal["spearman","pearson","kendall"]` | `"spearman"` | 相关性方法 |

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `*x` | `Factor` | 待测试的因子列表（至少 2 个） |
| `mask` | `Optional[Factor]` | 筛选条件因子 |
| `factor_name_list` | `Optional[List[str]]` | 因子名称列表 |

因子名称按升序排列后，两两组合生成截面 ID（格式为 `"因子A-因子B"`）。

### SectionCorrelation — 回测报告节点

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"因子截面相关性"` | 节点名称（冻结） |
| `FactorNameList` | `Optional[List[str]]` | `None` | 因子名称列表 |
| `GenReport` | `bool` | `False` | 自动生成报告 |

**输出**：时序截面相关性数据 + 平均相关性矩阵（带热力图着色）。

### 构造示例

```python
from QuantStudio.BackTest.SectionFactor.Correlation import CalcSectionCorrelation, SectionCorrelation

CorrFactor = CalcSectionCorrelation(descriptor_ids=SectionIDs)(*FactorList, mask=Mask)
CorrNode = SectionCorrelation(CorrFactor, args={"GenReport": True})
```

## Fama-MacBeth 回归

### CalcFamaMacBethRegression — 回归计算算子

`CalcFamaMacBethRegression` 是一个 `PanelOperator`，在每期进行截面回归。对每个因子做两种回归：
- **Pure（多元回归）**：所有测试因子 + 类别哑变量同时进入回归方程，得到的回归系数即为剥离其他因子影响后的纯因子收益
- **Raw（单变量回归）**：仅当前因子 + 类别哑变量进入回归方程，得到未剥离的原始因子收益

**构造参数**：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `descriptor_ids` | `List[str]` | — | 截面 ID 序列 |
| `lookback` | `int` | `31` | 时间标尺回溯期数 |
| `period_lookback` | `int` | `1` | 计算标尺回溯期数 |

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `*x` | `Factor` | 待测试的因子列表 |
| `price` | `Factor` | 价格因子 |
| `mask` | `Optional[Factor]` | 筛选条件因子 |
| `cat_data` | `Optional[Factor]` | 类别因子（作为哑变量进入回归） |
| `factor_name_list` | `Optional[List[str]]` | 因子名称列表 |

**返回值**：复合类型因子，每行包含 10 个字段：`PureReturn`、`PureT`、`PureF`、`PureR`、`PureRAdj`、`RawReturn`、`RawT`、`RawF`、`RawR`、`RawRAdj`。

### FamaMacBethRegression — 回测报告节点

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"Fama-MacBeth 回归"` | 节点名称（冻结） |
| `FactorNameList` | `Optional[List[str]]` | `None` | 因子名称列表 |
| `RollingAvgPeriod` | `int` | `12` | 滚动 t 统计量的计算期数 |
| `GenReport` | `bool` | `False` | 自动生成报告 |

**输出数据结构**：

| 键 | 说明 |
|------|------|
| `"Pure Return"` / `"Raw Return"` | 纯因子收益 / 原始因子收益时序 |
| `"回归t统计量(Pure)"` / `"(Raw)"` | 各期回归的 t 统计量 |
| `"回归F统计量(Pure)"` / `"(Raw)"` | 各期回归的 F 统计量 |
| `"滚动t统计量(Pure)"` / `"(Raw)"` | 滚动窗口计算的 t 统计量 |
| `"统计数据"` | 年化收益率、跟踪误差、信息比率、胜率、t 统计量（Pure/Raw/Pure-Raw） |
| `"回归统计量均值"` | 各统计指标的时间序列均值 |

### 构造示例

```python
from QuantStudio.BackTest.SectionFactor.ReturnDecomposition import CalcFamaMacBethRegression, FamaMacBethRegression

FMFactor = CalcFamaMacBethRegression(descriptor_ids=SectionIDs)(
    *FactorList, price=Price, mask=Mask, cat_data=Industry
)

FMNode = FamaMacBethRegression(FMFactor, args={"GenReport": True})
```

报告展示 Pure vs Raw 的年化收益率对比柱状图、t 统计量对比图以及 Pure-Raw 差异图。

## 因子换手率

### CalcFactorTurnover — 换手率计算算子

`CalcFactorTurnover` 是一个 `PanelOperator`，计算因子在前后两期的截面秩相关性（即 1 - 换手率）。换手率越低说明因子值越稳定。

**构造参数**：`descriptor_ids`、`lookback`、`period_lookback`、`corr_method`，与 `CalcIC` 含义一致。

**`__call__` 参数**：

| 参数 | 类型 | 说明 |
|------|------|------|
| `*x` | `Factor` | 待测试的因子列表 |
| `mask` | `Optional[Factor]` | 筛选条件因子 |
| `factor_name_list` | `Optional[List[str]]` | 因子名称列表 |

### FactorTurnover — 回测报告节点

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"因子换手率"` | 节点名称（冻结） |
| `FactorNameList` | `Optional[List[str]]` | `None` | 因子名称列表 |
| `GenReport` | `bool` | `False` | 自动生成报告 |

**输出统计数据**：各因子的换手率平均值、标准差、最小值、最大值、中位数，以及时序堆叠面积图。

### 构造示例

```python
from QuantStudio.BackTest.SectionFactor.Correlation import CalcFactorTurnover, FactorTurnover

TurnoverFactor = CalcFactorTurnover(
    descriptor_ids=SectionIDs, lookback=31, period_lookback=1
)(*FactorList, mask=Mask)

TurnoverNode = FactorTurnover(TurnoverFactor, args={"GenReport": True})
```

## 分位数组合

### makeQuantilePortfolio — 创建分位数组合

`makeQuantilePortfolio` 是一个工厂函数（非类），内部使用 `fo.SectionRank` 做截面排序分组，再通过 `CalcMaskPortfolio`（来自 `Strategy.AllocationStrategy`）为每组生成投资组合因子。

**参数**：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `factor` | `Factor` | — | 用于分组的因子对象 |
| `mask` | `Optional[Factor]` | `None` | 筛选条件因子 |
| `cat_data` | `Optional[Factor]` | `None` | 分类因子，非 None 时在每个类别内单独分组再汇总 |
| `weight` | `Optional[Factor]` | `None` | 权重因子，None 时等权 |
| `descriptor_ids` | `Optional[List[str]]` | `None` | 截面 ID 序列 |
| `rebalance_dts` | `Optional[List[datetime]]` | `None` | 再平衡时点序列 |
| `ascending` | `bool` | `False` | 是否升序排列 |
| `group_num` | `int` | `5` | 分组数（通常取 5 或 10） |
| `**kwargs` | — | — | 传递给 `CalcMaskPortfolio` 的额外参数 |

**返回值**：`List[SectionOperation]`，长度为 `group_num` 的分位数组合因子列表，每组为一个权重因子（截面和为 1）。

**内部实现**：
1. 调用 `fo.SectionRank(factor, mask=mask, cat_data=cat_data)` 计算截面排序（均匀化到 [0, 1)）
2. 对每组区间 `[i/group_num, (i+1)/group_num)` 创建 `CalcMaskPortfolio` 算子
3. 返回各组的权重因子

### MultiPortfolio — 多组合对比回测节点

`MultiPortfolio` 是 `BTNode` 子类，计算各组净值、超额收益率、风险指标，并生成对比图表。

**参数（`__QS_ArgClass__`）**：

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"多组合对比"` | 节点名称（冻结） |
| `LSPairs` | `List[Tuple[str,str]]` | `[]` | 多空组合对，如 `[('P4', 'P0')]` |
| `RebalanceDTs` | `Optional[List[datetime]]` | `None` | 再平衡时点，用于计算换手率 |
| `GenReport` | `bool` | `False` | 自动生成报告 |

**构造方法**：

```python
MultiPortfolio(
    nv: Factor,                    # 净值因子（由 CalcPortfolioNV 产生，列为各组净值）
    bmk_nv: Optional[Factor],      # 基准净值因子
    portfolio_list: Optional[List[Factor]],  # 各组的权重因子列表
    bmk_portfolio: Optional[Factor],         # 基准权重因子
    args={}, config_file=None, **kwargs
)
```

**输出统计指标**：年化收益率、波动率、Sharpe 比率、最大回撤、超额收益率、跟踪误差、信息比率、胜率、CAPM Alpha/Beta、平均换手率等。报告包含净值曲线、超额净值曲线、持仓数量图以及各统计指标的柱状图。

### 构造示例

```python
from QuantStudio.BackTest.SectionFactor.QuantilePortfolio import makeQuantilePortfolio, MultiPortfolio
from QuantStudio.BackTest.Strategy.AllocationStrategy import CalcPortfolioNV

# 创建分位数组合
QuantilePortfolios = makeQuantilePortfolio(
    FactorList[0],              # 按第一个因子分组
    mask=Mask,
    cat_data=Industry,
    weight=None,                # 等权
    descriptor_ids=SectionIDs,
    rebalance_dts=BalanceDTs,
    group_num=5
)

# 计算各组净值
calcPortfolioNV = CalcPortfolioNV(descriptor_ids=SectionIDs, start_dt=TestDTs[0])
PortfolioNV = calcPortfolioNV(*QuantilePortfolios, price=Price, init_nv=1)

# 创建多组合对比节点
QuantileNode = MultiPortfolio(
    nv=PortfolioNV,
    portfolio_list=QuantilePortfolios,
    args={"RebalanceDTs": BalanceDTs, "GenReport": True, "Name": "分位数组合"}
)
```

## 风险调整的 IC

`CalcRiskAdjustedIC` 算子先对因子暴露和收益率分别与给定的风险因子做正交化，然后计算正交化残差之间的 IC，以衡量因子在剔除风险因子影响后的纯 alpha 预测能力。

### CalcRiskAdjustedIC 构造参数

与 `CalcIC` 基本一致（`descriptor_ids`、`lookback`、`period_lookback`、`corr_method`），额外通过 `__call__` 接收风险因子。

### `__call__` 参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `*x` | `Factor` | 待测试的因子 |
| `price` | `Factor` | 价格因子 |
| `risk_factors` | `List[Factor]` | 风险因子列表，用于正交化 |
| `mask` | `Optional[Factor]` | 筛选条件因子 |
| `cat_data` | `Optional[Factor]` | 类别因子，非 None 时也参与正交化 |
| `factor_name_list` | `Optional[List[str]]` | 因子名称列表 |

### 构造示例

```python
from QuantStudio.BackTest.SectionFactor.IC import CalcRiskAdjustedIC, IC

# 假设 SizeFactor 是市值因子
RiskAdjustedIC = CalcRiskAdjustedIC(
    descriptor_ids=SectionIDs,
    lookback=31,
    period_lookback=1,
    corr_method="spearman"
)(
    *FactorList,
    price=Price,
    risk_factors=[SizeFactor],  # 风险因子列表
    mask=Mask,
    cat_data=Industry,
    factor_args={"CalcDTRuler": BalanceDTs}
)

RiskAdjICNode = IC(RiskAdjustedIC, args={"GenReport": True, "Name": "风险调整IC"})
```

> **注意**：`CalcRiskAdjustedIC` 与 `CalcIC` 共用同一个回测报告节点 `IC`，输出数据结构一致。

## IC 衰减

`ICDecay` 是回测节点，接收一组不同回溯期的 IC 因子，分析 IC 随回溯期数增加的变化趋势。适用于考察因子的预测能力随持有期延长而衰减的情况。

### ICDecay 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"IC 衰减"` | 节点名称（冻结） |
| `PeriodList` | `Optional[List[int]]` | `None` | 回溯期列表（如 `[1, 2, 3]` 表示 1 个月、2 个月、3 个月前因子值），None 时自动从 `ic_list` 长度生成 `range(len(ic_list))` |
| `FactorNameList` | `Optional[List[str]]` | `None` | 因子名称列表 |

### 输出数据结构

`ICDecay` 按因子拆分输出，每个因子包含：
- `"IC"` / `"Breadth"`：不同回溯期的 IC 和截面宽度
- `"统计数据"`：各回溯期的 IC 平均值、标准差、IC_IR、t 统计量、胜率

### 构造示例

```python
from QuantStudio.BackTest.SectionFactor.IC import CalcIC, ICDecay

# 创建不同 period_lookback 的 IC 因子列表
ICList = [
    CalcIC(descriptor_ids=SectionIDs, lookback=31*i, period_lookback=i)(
        *FactorList, price=Price, mask=Mask, cat_data=Industry,
        factor_args={"CalcDTRuler": BalanceDTs}
    )
    for i in range(1, 4)  # 1个月、2个月、3个月前因子值
]

# 创建 IC 衰减节点
ICDecayNode = ICDecay(ICList, args={"GenReport": True})
```

报告展示各回溯期 IC 均值的柱状图和胜率曲线。

## 完整示例：多维度截面因子测试

下面展示一个完整的截面因子测试流程，包含 IC、IC 衰减、分位数组合、因子换手率、截面相关性和 Fama-MacBeth 回归六个维度。

> **执行流程**（三层嵌套 `Cache → Context → Engine`）以及 `BTReport` 的报告汇总机制已在 **[基本框架](基本框架.ipynb)** 中详细说明，此处不再赘述。

In [2]:
# 参数设置
from QuantStudio.Factor.HDF5DB import HDF5DB
FDB = HDF5DB(args={"MainDir": Path(__QS_MainPath__).parent / "docs/data/HDF5"}).connect()

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 31)# 数据起止时间
TestStartDT, TestEndDT = dt.datetime(2025, 2, 28), EndDT# 测试起止时间

FT = FDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

# 再平衡时点序列
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime
BalanceDTs = getMonthLastDateTime(DTRuler)# 月末

In [ ]:
# 导入回测相关模块
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.BackTest.SectionFactor.IC import CalcIC, IC, ICDecay
from QuantStudio.BackTest.SectionFactor.QuantilePortfolio import makeQuantilePortfolio, MultiPortfolio
from QuantStudio.BackTest.Strategy.AllocationStrategy import CalcPortfolioNV
from QuantStudio.BackTest.SectionFactor.Correlation import CalcFactorTurnover, FactorTurnover, CalcSectionCorrelation, SectionCorrelation
from QuantStudio.BackTest.SectionFactor.ReturnDecomposition import CalcFamaMacBethRegression, FamaMacBethRegression

# 获取数据因子
FT = FDB.getTable("stock_cn_status")
Mask = (FT.getFactor("if_listed")==1)

FT = FDB.getTable("stock_cn_day_bar")
Price = FT.getFactor("close")

FT = FDB.getTable("stock_cn_industry")
Industry = FT.getFactor("industry")

FT = FDB.getTable("stock_cn_factor_value")
FactorList = [FT.getFactor(iFactorName) for iFactorName in ["bp_lr", "ep_ttm"]]

# ========== 构建回测节点列表 ==========
NodeList = []

# 1. Rank IC
FactorIC = CalcIC(descriptor_ids=SectionIDs, lookback=31, period_lookback=1, corr_method="spearman")(
    *FactorList, price=Price, mask=Mask, cat_data=Industry,
    factor_args={"CalcDTRuler": BalanceDTs}
)
ICNode = IC(FactorIC, args={"RollingAvgPeriod": 2, "GenReport": True})
NodeList.append(ICNode)

# 2. IC 衰减（1~2 个月回溯）
ICDecayNode = ICDecay(
    ic_list=[
        CalcIC(descriptor_ids=SectionIDs, lookback=31*i, period_lookback=i)(
            *FactorList, price=Price, mask=Mask, cat_data=Industry,
            factor_args={"CalcDTRuler": BalanceDTs}
        )
        for i in range(1, 3)
    ],
    args={"GenReport": True}
)
NodeList.append(ICDecayNode)

# 3. 分位数组合
calcPortfolioNV = CalcPortfolioNV(descriptor_ids=SectionIDs, start_dt=TestDTs[0])
for iFactor in FactorList:
    iQuantilePortfolioList = makeQuantilePortfolio(
        iFactor, mask=Mask, cat_data=Industry, weight=None,
        descriptor_ids=SectionIDs, rebalance_dts=BalanceDTs, group_num=5
    )
    iPortfolioNV = calcPortfolioNV(*iQuantilePortfolioList, price=Price, init_nv=1)
    iQuantilePortfolioNode = MultiPortfolio(
        nv=iPortfolioNV, portfolio_list=iQuantilePortfolioList,
        args={"RebalanceDTs": BalanceDTs, "GenReport": True, "Name": f"{iFactor.Name}-分位数组合"}
    )
    NodeList.append(iQuantilePortfolioNode)

# 4. 因子换手率
TurnoverFactor = CalcFactorTurnover(descriptor_ids=SectionIDs, lookback=31, period_lookback=1)(
    *FactorList, mask=Mask
)
FactorTurnoverNode = FactorTurnover(TurnoverFactor, args={"GenReport": True})
NodeList.append(FactorTurnoverNode)

# 5. 截面相关性
SectionCorrelationFactor = CalcSectionCorrelation(descriptor_ids=SectionIDs)(*FactorList, mask=Mask)
SectionCorrelationNode = SectionCorrelation(SectionCorrelationFactor, args={"GenReport": True})
NodeList.append(SectionCorrelationNode)

# 6. Fama-MacBeth 回归
FamaMacBethFactor = CalcFamaMacBethRegression(descriptor_ids=SectionIDs)(
    *FactorList, price=Price, mask=Mask, cat_data=Industry
)
FamaMacBethNode = FamaMacBethRegression(FamaMacBethFactor, args={"GenReport": True})
NodeList.append(FamaMacBethNode)

# ========== 创建报告并执行 ==========
Report = BTReport(bt_node_list=NodeList)

CacheDir = Path(__QS_MainPath__).parent / "docs/data/Cache"
if not CacheDir.exists(): CacheDir.mkdir(parents=True)

with FeatherFactorCache(args={"DTRuler": DTRuler, "PIDs": ["0"], "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(PID="0", PIDList=["0"], DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Output, = ExecEngine.run(
                [Report], Context,
                fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))]
            )

display(HTML(Output["Report"]))

## 输出解读

报告 HTML 按模块编号依次展示，每个模块包含参数设置说明、统计表格和可视化图表。

### 评价标准参考

| 指标 | 参考标准 |
|------|----------|
| **IC 均值** | 绝对值 > 2% 表明因子有较好的选股能力 |
| **IC_IR（信息比率）** | > 0.3 为较好，> 0.5 为优秀 |
| **t 统计量** | 绝对值 > 2 表明 IC 显著不为零 |
| **胜率** | > 55% 表明因子方向一致性好 |
| **分位数组合单调性** | 各组收益率随分位数单调变化，表明因子区分度高 |
| **因子换手率** | 月频换手率在 80%~95% 之间通常可接受，过低可能失效，过高交易成本大 |
| **截面相关性** | 因子间相关性 < 0.5 说明因子信息重叠度低 |
| **Fama-MacBeth Pure 收益** | t 统计量 > 2 表明剥离其他因子后仍具显著预测力 |

### 输出字典结构

`BTReport` 的 `backward_compute` 返回字典的各键可通过编程方式访问各模块的统计数据，例如：

```python
# 访问 IC 节点的完整输出
ic_output = Output["0-IC"]  # 键格式: "{序号}-{节点Name}"
ic_stats = ic_output["统计数据"]    # IC 汇总统计 DataFrame
ic_series = ic_output["IC"]         # IC 时序 DataFrame

# 访问分位数组合节点
portfolio_output = Output["2-bp_lr-分位数组合"]
portfolio_stats = portfolio_output["统计数据"]  # 绩效指标 DataFrame
portfolio_nv = portfolio_output["净值"]          # 各组净值 DataFrame
```